# 26 — Attention: scores, softmax, weighted sum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/026-attention.ipynb)

Attention lets one token pull numbers out of another. Each token asks a question (its query), every token advertises what it holds (its key and its value), and the answers are mixed in proportion to how well the question matched. Here you do that for two tokens by hand, then watch what the divisor sqrt(d_k) is protecting you from.

## 1. The numbers

Two tokens, two numbers each. Token 1 asks the question; both tokens offer a key and a value. The match score is a dot product, softened by dividing by the square root of the vector width, and the softmax turns the two scores into weights that sum to one.

In [ ]:
import numpy as np                                  # the array library, already on Colab

q = np.array([1., 2.])                             # token 1's query: what it is looking for
k1 = np.array([1., 0.])                            # token 1's key: what it advertises
k2 = np.array([0., 3.])                            # token 2's key
v1 = np.array([1., 1.])                            # token 1's value: what it hands over
v2 = np.array([0., 2.])                            # token 2's value
keys = np.stack([k1, k2])                          # both keys as rows of one array
values = np.stack([v1, v2])                        # both values as rows
dk = 2                                             # width of every vector

scores = keys @ q / np.sqrt(dk)                    # match scores, shrunk by 1/sqrt(dk)
e = np.exp(scores - scores.max())                  # exponentiate, after subtracting the max
weights = e / e.sum()                              # normalise so the weights sum to 1
out = weights @ values                             # the weighted sum of the values

print("scores  =", np.round(scores, 3))            # [0.707 4.243]
print("weights =", np.round(weights, 3))           # [0.03 0.97]
print("output  =", np.round(out, 3))               # mostly token 2's value
print("weights sum to", round(weights.sum(), 6))   # exactly 1.0

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — load NumPy for the array arithmetic.</li><li><code>q = ...</code> — the query of the token doing the looking.</li><li><code>k1</code>, <code>k2</code> — the keys of the two tokens; the query is compared to each.</li><li><code>v1</code>, <code>v2</code> — the values; these are the numbers that actually get mixed.</li><li><code>np.stack</code> — glue two vectors into a 2-row array so we can multiply them at once.</li><li><code>dk = 2</code> — the width of every vector; the divisor is built from it.</li><li><code>scores = ...</code> — dot product of the query with every key, then scale down.</li><li><code>e = ...</code> — exponentiate so bigger scores become bigger positive weights.</li><li><code>weights = ...</code> — divide by the total so the two weights add to one.</li><li><code>out = weights @ values</code> — one output vector: a blend of the two values.</li><li><code>print(...)</code> — the scores, the weights and the output, rounded for reading.</li><li><code>weights.sum()</code> — a check that the weights really do add to one.</li></ul>

</details>

## 2. Step by step

The matrix form hides the loop. Walk over the keys one at a time and keep a running output vector: each key contributes its value, scaled by the weight it earned.

In [ ]:
total = np.exp(scores - scores.max()).sum()        # the denominator of the softmax
acc = np.zeros(2)                                  # running output, starts at zero

for j in range(2):                                 # one pass per key
    e_j = np.exp(scores[j] - scores.max())         # this key's numerator
    w_j = e_j / total                              # this key's normalised weight
    acc = acc + w_j * values[j]                    # add its value, scaled by the weight
    print(f"key {j}: score={scores[j]:6.3f}  weight={w_j:.3f}")   # inspect each step

print("output     =", np.round(acc, 3))            # identical to section 1
print("same as the matrix form:", np.allclose(acc, out))   # a boolean, checked for you

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>total = ...</code> — the sum of all unnormalised weights, computed once.</li><li><code>acc = np.zeros(2)</code> — the output vector we are building up.</li><li><code>for j in range(2)</code> — visit key 1, then key 2.</li><li><code>e_j = ...</code> — this key's exponentiated score.</li><li><code>w_j = e_j / total</code> — turn it into a share of the whole.</li><li><code>acc = acc + w_j * values[j]</code> — accumulate the weighted value.</li><li><code>print(f"...")</code> — an f-string; the braces are filled with the values.</li><li><code>np.allclose(...)</code> — checks the loop landed on the same vector as section 1.</li></ul>

</details>

## 3. Plot: the attention weights as a heatmap

With two queries and two keys the weights form a 2-by-2 table: one row per query, one column per key. A dark cell means that query spends its weight on that key.

In [ ]:
import matplotlib.pyplot as plt                    # plotting library, already on Colab

Q = np.stack([[1., 2.], [2., 0.]])                 # two queries, one per token
W = np.zeros((2, 2))                               # the weight table we will fill
for i in range(2):                                 # one row per query
    s = keys @ Q[i] / np.sqrt(dk)                  # scores of this query against both keys
    e_row = np.exp(s - s.max())                    # stable exponentials
    W[i] = e_row / e_row.sum()                     # row of weights, sums to one

plt.figure(figsize=(4, 3))                         # a small square figure
plt.imshow(W, cmap="Blues", vmin=0, vmax=1)        # draw the table as colours
plt.xticks([0, 1], ["key 1", "key 2"])             # name the columns
plt.yticks([0, 1], ["query 1", "query 2"])         # name the rows
plt.colorbar(label="attention weight")             # a scale for the colours
plt.title("each query spends its weight where the key matches")   # the takeaway
plt.show()                                         # render it

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — load the plotting library.</li><li><code>Q = ...</code> — two different queries, one per token.</li><li><code>W = np.zeros((2, 2))</code> — an empty 2-by-2 table for the weights.</li><li><code>for i in range(2)</code> — fill one row of the table per query.</li><li><code>s = keys @ Q[i] / np.sqrt(dk)</code> — scores of this query against every key.</li><li><code>W[i] = e_row / e_row.sum()</code> — softmax over the row, stored in the table.</li><li><code>plt.imshow(...)</code> — draw the table as a heatmap; <code>vmin</code>/<code>vmax</code> fix the colour scale.</li><li><code>plt.colorbar(...)</code> — a legend for the colours.</li><li><code>e_row = ...</code> — stable exponentials for one row, computed just above.</li><li><code>plt.figure(...)</code> — a small canvas; <code>figsize</code> is inches.</li><li><code>plt.xticks</code> / <code>plt.yticks</code> — name the columns and the rows.</li><li><code>plt.title(...)</code> — the takeaway, not a restatement of the axes.</li><li><code>plt.show()</code> — render the figure.</li></ul>

</details>

## 4. Interactive: slide the vector width

Keep the two keys pointing in nearly the same direction and grow the width of the vectors. The dot products grow with the width, so the softmax keeps sharpening even though we already divided by sqrt(d_k).

In [ ]:
from ipywidgets import interact                    # sliders that rerun a function

def show(dk=2):                                    # dk comes from the slider
    q = np.ones(dk)                                # a query of all ones
    k1 = np.ones(dk)                               # key 1 points the same way
    k2 = np.linspace(0.2, 0.6, dk)                 # key 2 points a bit differently
    s = np.array([k1 @ q, k2 @ q]) / np.sqrt(dk)   # the two scaled scores
    e_s = np.exp(s - s.max())                      # stable exponentials
    p = e_s / e_s.sum()                            # the two weights
    plt.figure(figsize=(4, 3))                     # a fresh figure on every drag
    plt.bar(["key 1", "key 2"], p, color=["tab:blue", "tab:orange"])   # one bar per weight
    plt.ylim(0, 1)                                 # fixed axis so heights are comparable
    plt.ylabel("attention weight")                 # what the bars measure
    plt.title(f"d_k = {dk}: biggest weight {p.max():.3f}")   # live readout
    plt.show()                                     # draw the frame

interact(show, dk=(1, 64, 1))                      # attach the slider

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — Colab ships the widget library.</li><li><code>def show(dk=2)</code> — the function that draws one frame.</li><li><code>q = np.ones(dk)</code> — a query of length <code>dk</code>, all ones.</li><li><code>k2 = np.linspace(...)</code> — a second key that differs a little along every position.</li><li><code>s = ...</code> — both dot products, divided by <code>sqrt(dk)</code>.</li><li><code>p = e_s / e_s.sum()</code> — softmax over the two scores.</li><li><code>plt.bar(...)</code> — draw the two weights as bars.</li><li><code>plt.ylim(0, 1)</code> — fix the axis so the heights stay comparable across drags.</li><li><code>plt.ylabel(...)</code> — what the bars measure.</li><li><code>plt.title(...)</code> — the live readout of the largest weight.</li><li><code>interact(show, dk=(1, 64, 1))</code> — the slider, its range and step.</li></ul>

</details>

## 5. When it breaks: unscaled scores saturate

Forget the divisor and the scores are raw dot products, whose size grows with the vector width. A gap of a few units between two scores is enough to push one weight to almost one, and then the other token is ignored entirely.

In [ ]:
q = np.ones(8) * 2.                              # a wide query, all entries 2
k1 = np.ones(8) * 1.5                            # key 1, every entry 1.5
k2 = np.ones(8) * 1.0                            # key 2, every entry 1.0
raw = np.array([k1 @ q, k2 @ q])                 # dot products, no scaling at all
scaled = raw / np.sqrt(8)                        # the same scores, divided by sqrt(d_k)
p_raw = np.exp(raw - raw.max())                  # softmax of the raw scores
p_raw = p_raw / p_raw.sum()                      # normalise
p_sc = np.exp(scaled - scaled.max())             # softmax of the scaled scores
p_sc = p_sc / p_sc.sum()                         # normalise

print("raw scores   ", raw, "-> biggest weight", round(p_raw.max(), 4))    # 0.9997
print("scaled scores", np.round(scaled, 2), "-> biggest weight", round(p_sc.max(), 3))   # 0.944

plt.figure(figsize=(4, 3))                       # a short, wide figure
plt.bar(["unscaled", "scaled by 1/sqrt(d_k)"], [p_raw.max(), p_sc.max()],   # labels, then heights
        color=["tab:red", "tab:green"])          # one bar per version
plt.axhline(0.999, color="black", linestyle="--")   # the "one token wins" line
plt.ylim(0, 1.02)                                # keep the whole range visible
plt.ylabel("largest attention weight")           # what the bars measure
plt.title("unscaled: 0.9997 of the weight lands on one key")   # the takeaway
plt.show()                                       # render it

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>q = np.ones(8) * 2.</code> — a wide query; every entry is 2.</li><li><code>k1</code>, <code>k2</code> — two keys differing by 0.5 in every entry.</li><li><code>raw = ...</code> — the dot products with no divisor at all.</li><li><code>scaled = raw / np.sqrt(8)</code> — what the usual correction would have done.</li><li><code>p_raw</code>, <code>p_sc</code> — softmax of each, after subtracting the max for stability.</li><li><code>print(...)</code> — the two largest weights, side by side.</li><li><code>plt.bar(...)</code> — compare them as two bars.</li><li><code>plt.axhline(0.999, ...)</code> — a dashed line at the 'one token wins' level.</li><li><code>plt.ylim(0, 1.02)</code> — keep the whole 0-to-1 range visible.</li><li><code>plt.ylabel(...)</code> — what the bars measure.</li></ul>

</details>

## Your turn

In section 4, set the second key to `np.linspace(0.9, 1.0, dk)` and predict the largest weight before you drag. Then in section 5, change `0.5` to `0.2` for the gap between the keys — how wide must the vectors be before the unscaled softmax saturates again?